In [3]:
# basic chain
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model="llama3.2:latest")
prompt = ChatPromptTemplate(
    [
        ("system", "you are a helpful assistant. Answer briefly."),
        ("human", "{question}")
    ]
)

chain = prompt | llm
result = chain.invoke({"question": "What is a DAG?"})
print(result.content)

A Directed Acyclic Graph (DAG) is a type of graph where there are no directed cycles, meaning that it's not possible to start at a node and return to the same node by following the edges in the same direction. This makes DAGs useful for modeling workflows, dependencies, and other processes with distinct start and end points.


In [7]:
# structured output with pydantic
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    sentiment: str = Field(description="positive, negative or neutral")
    score: int = Field(description="Score from 1 to 10")

llm = ChatOllama(model="llama3.2:latest")

structured_llm = llm.with_structured_output(MovieReview)

review = structured_llm.invoke("Review Inception in a structured way")
print(review)



title='Inception Review' sentiment='positive' score=9


In [11]:
# tools
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get weather for a city (mock)."""
    return f"The weather in {city} is 22°C and sunny."

llm = ChatOllama(model="llama3.2:latest")

# bind the tools
llm_with_tools = llm.bind_tools([multiply, get_weather])

response = llm_with_tools.invoke("What is 6 times 9?")
print(response.tool_calls)
print(response)

[{'name': 'multiply', 'args': {'a': '6', 'b': '9'}, 'id': '82210f66-8203-49e0-8d1c-03d0be42fc40', 'type': 'tool_call'}]
content='' additional_kwargs={} response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-03-25T13:28:28.990923Z', 'done': True, 'done_reason': 'stop', 'total_duration': 592174667, 'load_duration': 73877542, 'prompt_eval_count': 190, 'prompt_eval_duration': 140871542, 'eval_count': 22, 'eval_duration': 368164003, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'} id='lc_run--019d252e-a9ed-7191-be1d-a902e947d7d8-0' tool_calls=[{'name': 'multiply', 'args': {'a': '6', 'b': '9'}, 'id': '82210f66-8203-49e0-8d1c-03d0be42fc40', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 190, 'output_tokens': 22, 'total_tokens': 212}


In [12]:
# lang graph linear flow
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, END

class State(TypedDict):
    question: str
    answer: str

def ask_llm(state: State) -> dict:
    llm = ChatOllama(model="llama3.2:latest")
    response = llm.invoke(state["question"])
    return {"answer": response.content}

builder = StateGraph(State)
builder.add_node("ask_llm", ask_llm)
builder.set_entry_point("ask_llm")
builder.add_edge("ask_llm", END)

graph = builder.compile()

result = graph.invoke({"question": "What is a planet?"})
print(result)

{'question': 'What is a planet?', 'answer': "A planet is a celestial body that orbits around a star, typically being large enough to be rounded by its own gravity and having sufficient mass to clear the surrounding space of other objects. The International Astronomical Union (IAU) defines a planet as:\n\n1. It must be in orbit around the Sun.\n2. It must be massive enough to be rounded by its own gravity (i.e., not irregularly shaped).\n3. It must have cleared the neighborhood around its orbit, meaning it is the dominant object in its orbit and has removed or ejected all other objects in its vicinity.\n\nBased on this definition, there are eight planets in our solar system:\n\n1. Mercury\n2. Venus\n3. Earth\n4. Mars\n5. Jupiter\n6. Saturn\n7. Uranus\n8. Neptune\n\nHowever, it's worth noting that Pluto was previously considered a planet but is now classified as a dwarf planet, which is a celestial body that meets criteria 1 and 2 but not criterion 3.\n\nHere are some key characteristics